# Standalone UCP MILP construction

`ucp_mip.py` parses a publication instance and constructs a solver-neutral MILP. It can either create a Gurobi model or solve the same model through `scipy.optimize.milp`, which uses HiGHS. Start Jupyter from this repository's root. If the kernel starts in `code/`, setup moves its working directory to the repository root; all data paths below are relative to that root.

In [ ]:
from pathlib import Path
import os
import sys

# Jupyter may start the kernel in the notebook's code/ directory.
if Path.cwd().name == 'code' and Path('ucp_mip.py').is_file():
    os.chdir(Path.cwd().parent)
REPO_ROOT = Path('.')
if not all((REPO_ROOT / name).is_dir() for name in ('code', 'instances', 'results')):
    raise FileNotFoundError("Start Jupyter from this repository's root directory.")

CODE_DIR = REPO_ROOT / 'code'
if str(CODE_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(CODE_DIR.resolve()))

from ucp_mip import (
    build_gurobi_model,
    build_mip,
    read_instance,
    solution_from_gurobi,
    solve_with_scipy,
)

In [ ]:
INSTANCE_PATH = REPO_ROOT / 'instances' / 'p_ucp_u2_t3_v1'
instance = read_instance(INSTANCE_PATH)
problem = build_mip(instance)
print(instance)
print(f'variables={problem.num_variables}, constraints={problem.num_constraints}')

## Solve with SciPy/HiGHS

In [ ]:
scipy_solution = solve_with_scipy(
    problem,
    time_limit=300.0,
    mip_rel_gap=0.0,
)
print('objective:', scipy_solution.objective_function_value)
print('MIP gap:', scipy_solution.mip_gap)
print('Solver status:', scipy_solution.solver_status, scipy_solution.solver_message)
list(scipy_solution.decision_variables.items())

## Build and solve the equivalent Gurobi model

This cell requires `gurobipy` and a usable Gurobi license. The returned `model` may also be exported with `model.write('publication_ucp.mps')`.

In [ ]:
try:
    model, gurobi_variables, gurobi_problem = build_gurobi_model(problem)
    model.Params.OutputFlag = 0
    model.Params.MIPGap = 0.0
    model.optimize()
    if model.SolCount == 0:
        raise RuntimeError(f'Gurobi returned no feasible solution (status {model.Status}).')
    gurobi_solution = solution_from_gurobi(
        gurobi_problem, gurobi_variables, model
    )
    print('Gurobi objective:', gurobi_solution.objective_function_value)
    print('Gurobi status:', gurobi_solution.solver_status)
    print('Gurobi achieved MIP gap:', gurobi_solution.mip_gap)
    print(
        'Difference from SciPy:',
        gurobi_solution.objective_function_value
        - scipy_solution.objective_function_value,
    )
except ImportError:
    print('gurobipy is not installed; the SciPy/HiGHS example above is complete.')